In [4]:
%pip install "tensorflow-hub<0.17" pillow matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

print("Libraries imported successfully!")

C:\Users\lenovo\AppData\Roaming\Python\Python313\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version



Libraries imported successfully!


In [3]:
print("Loading pretrained model...")

model_url = "https://tfhub.dev/tensorflow/efficientdet/lite0/detection/1"
detector = hub.load(model_url)

detect_fn = detector.signatures["serving_default"]

print("Pretrained model loaded successfully!")

Loading pretrained model...



ValueError: Trying to load a model of incompatible/unknown type. 'C:\Users\lenovo\AppData\Local\Temp\tfhub_modules\c3dc9e95fbaa581ada8f233917185874285fcfa8' contains neither 'saved_model.pb' nor 'saved_model.pbtxt'.

In [ ]:
image_path = "cat.jpg"

if not os.path.exists(image_path):
    raise FileNotFoundError("Image not found: " + image_path)

image = Image.open(image_path).convert("RGB")
image = image.resize((512, 512))

image_array = np.array(image, dtype=np.uint8)

print("cat image loaded successfully!")

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image_array)
plt.title("Original Image")
plt.axis("off")
plt.show()


In [ ]:
input_tensor = tf.convert_to_tensor(
    image_array,
    dtype=tf.uint8
)

input_tensor = tf.expand_dims(input_tensor, axis=0)

In [ ]:
print("Detecting objects...")

result = detect_fn(input_tensor)

print("Object detection completed!")

In [ ]:
boxes = result["output_0"][0].numpy()
scores = result["output_1"][0].numpy()
classes = result["output_2"][0].numpy()
num_detections = int(result["output_3"][0].numpy())

print("Detection results extracted successfully!")

In [ ]:
height, width, _ = image_array.shape

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image_array)

detected_count = 0

for i in range(min(num_detections, len(scores))):

    score = float(scores[i])

    if score < 0.30:
        continue


    ymin, xmin, ymax, xmax = boxes[i]

    left = max(0, min(int(xmin), width - 1))
    top = max(0, min(int(ymin), height - 1))
    right = max(0, min(int(xmax), width))
    bottom = max(0, min(int(ymax), height))


    if right <= left or bottom <= top:
        continue

 
    rectangle = plt.Rectangle(
        (left, top),
        right - left,
        bottom - top,
        fill=False,
        edgecolor="red",
        linewidth=2
    )

    ax.add_patch(rectangle)

    class_id = int(classes[i])

    label = "Class {}: {:.2f}".format(class_id, score)

    ax.text(
        left,
        max(top - 5, 0),
        label,
        color="white",
        fontsize=9,
        bbox=dict(facecolor="red", alpha=0.8)
    )

    detected_count += 1

    print(
        "Object:", detected_count,
        "| Class ID:", class_id,
        "| Confidence:", round(score, 3)
    )



In [ ]:
ax.set_title("Object Detection Using EfficientDet-Lite0")
ax.axis("off")
plt.tight_layout()
plt.show()

print("\nTotal objects detected:", detected_count)

if detected_count == 0:
    print("No objects detected above the confidence threshold.")

print("\nCONCLUSION:")
print(
    "Object detection was implemented using the pretrained "
    "EfficientDet-Lite0 CNN model. Bounding boxes and confidence "
    "scores were displayed for detected objects."
)